# 🚗 KnightSight EdgeVision — ANPR Pipeline
### Advanced Number Plate Recognition | Edge-Optimized | 100% Offline

---


## ⚙️ Step 1: Install Dependencies
Install all required libraries. This takes ~2-3 minutes on first run.

In [ ]:
# Install all required packages
!pip install ultralytics>=8.3.0 -q
!pip install paddlepaddle-gpu paddleocr -q  # GPU version
!pip install streamlit opencv-python-headless numpy pandas -q

print('✅ All dependencies installed successfully!')

---
## 📥 Step 2: Clone Project & Download Base Models
Pull the latest code from GitHub and download the YOLO11n base model.

In [ ]:
# Clone the project repository
!git clone https://github.com/kg2655/Deepsight-Hackathon.git
%cd Deepsight-Hackathon

# Download YOLO11n base model (auto-downloads ~5.4 MB)
from ultralytics import YOLO
vehicle_model = YOLO('yolo11n.pt')
print('✅ YOLO11n vehicle model loaded!')
print(f'   Parameters: 2.6M | GFLOPs: 6.5 | Size: ~5.4 MB')

---
## 📂 Step 2b: Get the Custom Plate Model (best.pt)
> `best.pt` is not on GitHub because it's too large. We need it to read plates.

**HOW TO DO THIS:**
1. Download `best.pt` from WhatsApp to your computer.
2. On the left side of this Colab page, click the **Folder icon 📁**.
3. **Drag and drop** `best.pt` into that sidebar to upload it to Colab.
4. Wait for it to finish uploading, then run this cell below.

In [ ]:
import os

DEST = 'runs/detect/runs/detect/plate_detector_yolo11/weights/'
os.makedirs(DEST, exist_ok=True)

# Check if uploaded via sidebar (it goes to /content/)
if os.path.exists('/content/best.pt'):
    os.system(f'mv /content/best.pt {DEST}')
    print("✅ Found best.pt in Colab sidebar! Moved it to the correct folder.")
else:
    print("⚠️ best.pt NOT FOUND in the sidebar!")
    print("Please drag and drop best.pt into the left folder sidebar, wait for it to upload, and run this cell again.")

---
## 🔧 Step 3: Setup Preprocessing Pipeline

In [ ]:
import cv2
import numpy as np

def preprocess_frame(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.merge((l, a, b))
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

    mean_val = np.mean(enhanced)
    if mean_val > 5:
        gamma = float(np.clip(np.log(127.5) / np.log(mean_val + 1e-3), 0.5, 2.0))
        table = np.array([((i / 255.0) ** (1.0/gamma)) * 255 for i in range(256)]).astype('uint8')
        enhanced = cv2.LUT(enhanced, table)

    blur = cv2.GaussianBlur(enhanced, (0, 0), 2.0)
    enhanced = cv2.addWeighted(enhanced, 1.4, blur, -0.4, 0)
    return enhanced
print('✅ Preprocessing pipeline ready!')

---
## ⚡ Step 4: Speed Benchmark
Run 30 inference passes and measure real latency to show judges.

In [ ]:
import torch
import time

device = 'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️'
print(f'Running benchmark on: {device}\n')

dummy = np.zeros((640, 640, 3), dtype=np.uint8)
times = []
VEHICLE_CLASSES = [2, 3, 5, 7]

for _ in range(5): vehicle_model(dummy, verbose=False) # Warmup

for i in range(30):
    t = time.perf_counter()
    vehicle_model(dummy, classes=VEHICLE_CLASSES, verbose=False)
    times.append((time.perf_counter() - t) * 1000)

print(f'   Mean latency:    {np.mean(times):.1f} ms')
print(f'   Throughput:      {1000/np.mean(times):.1f} FPS')
print(f'   250ms limit:     {"✅ PASS" if np.mean(times) < 250 else "❌ FAIL"}')

---
## 🌐 Step 5: Launch Streamlit App
Run the full interactive web demo. Open the `ngrok` link to share with judges.

In [ ]:
# Install ngrok tunnel
!pip install pyngrok -q
from pyngrok import ngrok

# Start the Streamlit app in background
!streamlit run app.py --server.port 8501 &>/dev/null&

time.sleep(4)
public_url = ngrok.connect(8501)
print('='*55)
print('🌐 STREAMLIT APP IS LIVE!')
print(f'👉 CLICK HERE TO OPEN THE APP: {public_url}')
print('='*55)
